# Does my causal forest beat OLS on a confounded + heterogeneous DGP?

This notebook uses `spuriosity` to generate data with **both** a confounded treatment (an observed variable, `age`, drives both program assignment and the outcome) **and** a genuinely nonlinear heterogeneous treatment effect (the program helps middle-aged participants most, tapering off at both ends — an inverted-parabola shape).

We compare two ways of estimating the treatment effect as a function of age:
1. **Naive OLS with a linear interaction term** (`age * program`) — a common, simple approach
2. **`econml`'s `CausalForestDML`** — a nonparametric method that makes no assumption about the shape of the effect

Because a linear interaction term can only represent a *straight-line* relationship between age and the effect, it should fail to capture the true peaked shape — while a causal forest, which makes no linearity assumption, should recover it much more accurately.

## Setup

In [2]:
%pip install -q "spuriosity[econml] @ git+https://github.com/Nityahapani/spuriosity.git"

Note: you may need to restart the kernel to use updated packages.


In [3]:
from spuriosity import PanelGenerator, reference
import numpy as np

print("spuriosity imported successfully")

spuriosity imported successfully


## Generate confounded + heterogeneous data

- `age` is an observed confounder (`add_confounder(..., observed=True)`) affecting both the outcome directly and being correlated with treatment assignment likelihood
- The true treatment effect is `8 - 0.01*(age-40)**2` — a downward parabola peaking at age 40

Note: the first `causal_forest_fit` call in a fresh environment can take ~15-20 seconds due to a one-time JIT compilation step in one of `econml`'s dependencies (`numba`); subsequent calls are fast.

In [5]:
gen = PanelGenerator(n_entities=10000, n_periods=1, seed=42)
gen.add_variable("age", dist="normal", mean=40, std=10)
gen.add_treatment("program", propensity=0.5, start_period=0)
gen.set_outcome(
    formula="age + program",
    coefficients={"age": 0.5, "program": 0.0, "Intercept": 10.0},
    noise_std=1.0,
)
gen.add_hte(treatment="program", modifier="age", formula="8 - 0.01*(age-40)**2")
gen.add_confounder(feature="age", outcome="y", strength=0.3, observed=True)
df, truth = gen.generate()
df.head()

,entity_id,period,age,program,_confounder_age,y
0,0,0,44.147570,1,-0.119100,38.216632
1,1,0,46.288084,1,0.774406,42.386963
2,2,0,40.294521,0,0.022141,30.877374
3,3,0,29.679957,1,1.741390,31.468297
4,4,0,54.413715,0,-0.761648,37.462232


In [6]:
print(truth)

GroundTruth(seed=42, spuriosity='0.1.0')
  true_coefficients: {'age': 0.5, 'program': 0.0, 'Intercept': 10.0}
  confounding_strength: {'age': 0.3}
  treatment_effect_ate: 7.0259
  has_true_cate: True


## Naive approach: OLS with a linear interaction

In [8]:
fit_ols = reference.ols_fit(df, formula="y ~ age*program + _confounder_age")
print("OLS coefficients:", fit_ols.coefficients)

def naive_cate(age_value):
    return fit_ols.coefficients.get("program", 0.0) + fit_ols.coefficients.get("age:program", 0.0) * age_value

OLS coefficients: {'Intercept': 10.033989278074989, 'age': 0.4991819696972295, 'program': 6.964591883443954, 'age:program': 0.0011527067015268053, '_confounder_age': 0.28916579473785886}


## Causal forest: nonparametric CATE estimation

In [10]:
fit_cf = reference.causal_forest_fit(
    df, outcome="y", treatment="program", covariates=["age", "_confounder_age"]
)
print("Causal forest ATE estimate:", round(fit_cf.ate_estimate, 3))

Causal forest ATE estimate: 7.045


## Compare CATE recovery at specific ages

In [12]:
test_ages = [20, 30, 40, 50, 60]
print(f"{'age':>5} {'true CATE':>12} {'naive OLS':>12} {'causal forest':>15}")
for age in test_ages:
    true_val = truth.true_cate(age)
    naive_val = naive_cate(age)
    cf_val = fit_cf.raw_model.effect(np.array([[age, 0.0]]))[0]
    print(f"{age:>5} {true_val:>12.3f} {naive_val:>12.3f} {cf_val:>15.3f}")

  age    true CATE    naive OLS   causal forest
   20        4.000        6.988           4.277
   30        7.000        6.999           7.079
   40        8.000        7.011           7.717
   50        7.000        7.022           7.082
   60        4.000        7.034           3.642


Notice that the naive OLS estimate barely changes across ages (it can only fit a straight line), while the causal forest tracks the true peaked shape much more closely.

## Aggregate error comparison

In [15]:
true_cate_per_row = df["age"].apply(truth.true_cate).to_numpy()
naive_cate_per_row = naive_cate(df["age"].to_numpy())
cf_cate_per_row = reference.causal_forest_predict(fit_cf, df)

naive_rmse = np.sqrt(np.mean((naive_cate_per_row - true_cate_per_row) ** 2))
cf_rmse = np.sqrt(np.mean((cf_cate_per_row - true_cate_per_row) ** 2))

print(f"Naive OLS-with-interaction CATE RMSE:  {naive_rmse:.3f}")
print(f"Causal forest CATE RMSE:               {cf_rmse:.3f}")

Naive OLS-with-interaction CATE RMSE:  1.426
Causal forest CATE RMSE:               0.276


## Takeaway

The true effect peaks for middle-aged participants and tapers off at both ends — a shape no *linear* interaction term can represent. The causal forest, which makes no linearity assumption, recovers this shape far more accurately, with meaningfully lower CATE RMSE.

This is the concrete case for reaching for a causal forest instead of a simple interaction model: when you suspect the treatment effect varies with a covariate in a genuinely nonlinear way, and you don't know the functional form in advance.